# Run All Data Processors

This notebook runs all data processing scripts to convert raw weather data to standardized formats.

## Available Processors:
1. **ASOS 1-Minute** - Process raw ASOS 1-minute data (`process_asos_1min.py`)
   - Input: `src/data/noaa_asos/noaa_asos_1min/raw/*_raw_*.csv`
   - Output: `src/data/output/asos_1min/`

2. **NOAA Daily** - Process NOAA daily data (`process_daily_noaa.py`)
   - Input: `src/data/noaa_asos/daily/417*.csv`
   - Output: `src/data/output/noaa_daily/`

3. **NYC Mesonet** - Process NYC Mesonet 5-minute data (`process_nycmesonet.py`)
   - Input: `src/data/Mesonet/*.zip`
   - Output: `src/data/output/mesonet/`

## Notes:
- **File handling**: If output files already exist, the scripts will prompt for confirmation:
  - `[r]eplace` - Overwrites existing files
  - `[a]dd` - Saves to a new timestamped folder (e.g., `output_dir/20240101_120000/`)
  - `[s]kip` - Skips processing
  - Set `SKIP_CONFIRMATIONS = True` to use default action from config
- The notebook checks for input data availability before running each processor
- All outputs follow standard data conventions (see `config.conventions`)

In [2]:
# Simple setup - just import config!
import subprocess
import warnings
warnings.filterwarnings('ignore')

from config_process import (
    BASE_PATH,
    ASOS_1MIN_DIR,
    NOAA_DAILY_DIR,
    MESONET_DIR,
    OUTPUT_DIR
)

# Get directories
from pathlib import Path
project_root = BASE_PATH
src_dir = project_root / "src"
process_dir = src_dir / "process"

print("✓ Setup complete")
print(f"Project root: {project_root}")
print(f"Output directory: {OUTPUT_DIR}")

✓ Setup complete
Project root: /Users/drorjac/OpenMesh_pynncml
Output directory: /Users/drorjac/OpenMesh_pynncml/src/data/output


## Configuration

Select which processors to run:

In [3]:
# Configuration: Select which processors to run
RUN_ASOS_1MIN = True      # Process ASOS 1-minute data
RUN_NOAA_DAILY = True     # Process NOAA daily data
RUN_MESONET = True        # Process NYC Mesonet data

# Optional: Skip confirmation prompts (set to False to get interactive prompts)
SKIP_CONFIRMATIONS = False  # If True, will not overwrite existing files

print("Configuration:")
print(f"  Run ASOS 1-Min:     {RUN_ASOS_1MIN}")
print(f"  Run NOAA Daily:     {RUN_NOAA_DAILY}")
print(f"  Run NYC Mesonet:    {RUN_MESONET}")
print(f"  Skip confirmations: {SKIP_CONFIRMATIONS}")

Configuration:
  Run ASOS 1-Min:     True
  Run NOAA Daily:     True
  Run NYC Mesonet:    True
  Skip confirmations: False


## Check Input Data Availability

Verify that input data exists before processing:

In [4]:
# Check input data availability
print("Checking input data availability...")
print("=" * 60)

data_status = {}

# Check ASOS 1-minute raw data
if RUN_ASOS_1MIN:
    raw_dir = ASOS_1MIN_DIR / "raw"
    raw_files = list(raw_dir.glob("*_raw_*.csv")) if raw_dir.exists() else []
    data_status['asos_1min'] = {
        'path': raw_dir,
        'exists': raw_dir.exists(),
        'files': len(raw_files),
        'file_list': [f.name for f in raw_files[:5]]  # Show first 5
    }
    if raw_files:
        print(f"✓ ASOS 1-Min: Found {len(raw_files)} raw file(s) in {raw_dir}")
        if len(raw_files) <= 5:
            for f in raw_files:
                print(f"    - {f.name}")
        else:
            for f in raw_files[:3]:
                print(f"    - {f.name}")
            print(f"    ... and {len(raw_files) - 3} more")
    else:
        print(f"✗ ASOS 1-Min: No raw files found in {raw_dir}")

# Check NOAA Daily data
if RUN_NOAA_DAILY:
    daily_files = list(NOAA_DAILY_DIR.glob("417*.csv")) if NOAA_DAILY_DIR.exists() else []
    data_status['noaa_daily'] = {
        'path': NOAA_DAILY_DIR,
        'exists': NOAA_DAILY_DIR.exists(),
        'files': len(daily_files),
        'file_list': [f.name for f in daily_files[:5]]
    }
    if daily_files:
        print(f"✓ NOAA Daily: Found {len(daily_files)} file(s) in {NOAA_DAILY_DIR}")
        if len(daily_files) <= 5:
            for f in daily_files:
                print(f"    - {f.name}")
        else:
            for f in daily_files[:3]:
                print(f"    - {f.name}")
            print(f"    ... and {len(daily_files) - 3} more")
    else:
        print(f"✗ NOAA Daily: No 417*.csv files found in {NOAA_DAILY_DIR}")

# Check NYC Mesonet data
if RUN_MESONET:
    zip_files = list(MESONET_DIR.glob("*.zip")) if MESONET_DIR.exists() else []
    data_status['mesonet'] = {
        'path': MESONET_DIR,
        'exists': MESONET_DIR.exists(),
        'files': len(zip_files),
        'file_list': [f.name for f in zip_files]
    }
    if zip_files:
        print(f"✓ NYC Mesonet: Found {len(zip_files)} zip file(s) in {MESONET_DIR}")
        for f in zip_files:
            print(f"    - {f.name}")
    else:
        print(f"✗ NYC Mesonet: No .zip files found in {MESONET_DIR}")

print("=" * 60)

Checking input data availability...
✓ ASOS 1-Min: Found 3 raw file(s) in /Users/drorjac/OpenMesh_pynncml/src/data/noaa_asos/noaa_asos_1min/raw
    - NYC_raw_20230801_20251110.csv
    - LGA_raw_20230801_20251110.csv
    - JFK_raw_20230801_20251110.csv
✓ NOAA Daily: Found 2 file(s) in /Users/drorjac/OpenMesh_pynncml/src/data/noaa_asos/daily
    - 4177732.csv
    - 4177747.csv
✓ NYC Mesonet: Found 1 zip file(s) in /Users/drorjac/OpenMesh_pynncml/src/data/Mesonet
    - request.zip


## Run Processors

Execute the selected processing scripts:

In [4]:
import sys

def run_processor(script_name, description, skip_confirm=False):
    """Run a processing script and capture output."""
    script_path = process_dir / script_name
    if not script_path.exists():
        print(f"✗ ERROR: Script not found: {script_path}")
        return False
    
    output_subdirs = {
        'process_asos_1min.py': 'asos_1min',
        'process_daily_noaa.py': 'noaa_daily',
        'process_nycmesonet.py': 'mesonet'
    }
    
    output_subdir = output_subdirs.get(script_name)
    action = None
    
    # Check for existing files
    if output_subdir:
        output_dir = OUTPUT_DIR / output_subdir
        if output_dir.exists():
            existing_files = [f for f in output_dir.glob("*.csv") 
                            if 'metadata' not in f.name.lower()]
            if existing_files and not skip_confirm:
                print(f"\n⚠ Files exist ({len(existing_files)} files)")
                while True:
                    response = input("  [r]eplace, [a]dd, or [s]kip? (r/a/s): ").strip().lower()
                    if response in ['r', 'replace']:
                        action = 'replace'
                        break
                    elif response in ['a', 'add']:
                        action = 'add'
                        break
                    elif response in ['s', 'skip']:
                        action = 'skip'
                        print("  → Skipping processing")
                        return None
                    else:
                        print("  Invalid. Enter 'r', 'a', or 's'.")
    
    # Use default from config if no choice made
    if action is None:
        from config_process import DEFAULT_PROCESSING_ACTION_IF_EXISTS
        action = DEFAULT_PROCESSING_ACTION_IF_EXISTS
    
    print(f"\n{'=' * 60}")
    print(f"Running: {description}")
    print(f"{'=' * 60}")
    
    try:
        result = subprocess.run(
            [sys.executable, str(script_path), '--action', action],
            cwd=str(process_dir),
            capture_output=True,
            text=True,
            timeout=3600
        )
        
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr)
        
        return result.returncode == 0
            
    except Exception as e:
        print(f"✗ ERROR: {e}")
        return False


# Track results
results = {}

# Run ASOS 1-Minute processor
if RUN_ASOS_1MIN:
    if data_status.get('asos_1min', {}).get('files', 0) > 0:
        results['asos_1min'] = run_processor(
            'process_asos_1min.py',
            'ASOS 1-Minute Data Processor',
            skip_confirm=SKIP_CONFIRMATIONS
        )
    else:
        print("\n⚠ Skipping ASOS 1-Min: No input data found")
        results['asos_1min'] = None

# Run NOAA Daily processor
if RUN_NOAA_DAILY:
    if data_status.get('noaa_daily', {}).get('files', 0) > 0:
        results['noaa_daily'] = run_processor(
            'process_daily_noaa.py',
            'NOAA Daily Data Processor',
            skip_confirm=SKIP_CONFIRMATIONS
        )
    else:
        print("\n⚠ Skipping NOAA Daily: No input data found")
        results['noaa_daily'] = None

# Run NYC Mesonet processor
if RUN_MESONET:
    if data_status.get('mesonet', {}).get('files', 0) > 0:
        results['mesonet'] = run_processor(
            'process_nycmesonet.py',
            'NYC Mesonet Data Processor',
            skip_confirm=SKIP_CONFIRMATIONS
        )
    else:
        print("\n⚠ Skipping NYC Mesonet: No input data found")
        results['mesonet'] = None


⚠ Files exist (3 files)
  Invalid. Enter 'r', 'a', or 's'.
  Invalid. Enter 'r', 'a', or 's'.

Running: ASOS 1-Minute Data Processor
ASOS 1-MINUTE DATA PROCESSOR
Input:  /Users/drorjac/OpenMesh_pynncml/src/data/noaa_asos/noaa_asos_1min/raw
Output: /Users/drorjac/OpenMesh_pynncml/src/data/output/asos_1min

Loading raw data from: /Users/drorjac/OpenMesh_pynncml/src/data/noaa_asos/noaa_asos_1min/raw
  ✓ JFK_raw_20230801_20251110.csv (1,069,614 rows)
  ✓ LGA_raw_20230801_20251110.csv (861,988 rows)
  ✓ NYC_raw_20230801_20251110.csv (888,254 rows)
✓ Loaded 3 stations

Processing to standard format...
  JFK: 1,069,614 rows, total precip = 2359.2 mm
  LGA: 861,988 rows, total precip = 2210.1 mm
  NYC: 888,254 rows, total precip = 2302.3 mm
✓ Processing complete

Checking format...
  ✓ JFK: OK
  ✓ LGA: OK
  ✓ NYC: OK
✓ All valid

Files exist in output directory. Saving to timestamped folder: /Users/drorjac/OpenMesh_pynncml/src/data/output/asos_1min/20251214_141931
Saving to: /Users/drorjac/Op

KeyboardInterrupt: 

## Data Format Summary

Analysis of input and output data formats - shared columns, unique columns, and standardization status.

In [ ]:
# Analyze data formats from actual output files
import pandas as pd
from config import STANDARD_COLUMNS, SOURCE_SPECIFIC_COLUMNS

print("=" * 60)
print("DATA FORMAT ANALYSIS")
print("=" * 60)

# Expected standard columns
standard_cols = list(STANDARD_COLUMNS.keys())
print(f"\nStandard columns (from config.conventions): {len(standard_cols)}")
print(f"  {', '.join(standard_cols)}")

# Read actual columns from output files
datasets = {}
output_dirs = {
    'ASOS 1-Min': OUTPUT_DIR / 'asos_1min',
    'NOAA Daily': OUTPUT_DIR / 'noaa_daily',
    'NYC Mesonet': OUTPUT_DIR / 'mesonet'
}

print("\n" + "=" * 60)
print("OUTPUT FILE COLUMNS")
print("=" * 60)

for name, out_dir in output_dirs.items():
    if out_dir.exists():
        csv_files = list(out_dir.glob("*.csv"))
        # Skip metadata files
        csv_files = [f for f in csv_files if 'metadata' not in f.name.lower()]
        if csv_files:
            # Read first file to get columns
            try:
                df = pd.read_csv(csv_files[0], nrows=1)
                datasets[name] = {
                    'columns': list(df.columns),
                    'file': csv_files[0].name,
                    'exists': True,
                    'file_count': len(csv_files)
                }
                cols = datasets[name]['columns']
                print(f"\n{name} ({datasets[name]['file']}):")
                print(f"  Files: {datasets[name]['file_count']}, Columns: {len(cols)}")
                
                # Identify standard vs source-specific
                standard_in_data = [c for c in cols if c in standard_cols]
                source_specific = [c for c in cols if c not in standard_cols and c != 'time']
                
                if standard_in_data:
                    print(f"  ✓ Standard ({len(standard_in_data)}): {', '.join(standard_in_data)}")
                if source_specific:
                    print(f"  • Source-specific ({len(source_specific)}): {', '.join(source_specific[:10])}")
                    if len(source_specific) > 10:
                        print(f"    ... and {len(source_specific) - 10} more")
            except Exception as e:
                datasets[name] = {'exists': True, 'error': str(e)}
                print(f"\n{name}: ERROR reading file - {e}")
        else:
            datasets[name] = {'exists': False, 'columns': []}
            print(f"\n{name}: No output files found")
    else:
        datasets[name] = {'exists': False, 'columns': []}
        print(f"\n{name}: Output directory does not exist")

# Find shared columns across all datasets
print("\n" + "=" * 60)
print("SHARED COLUMNS ANALYSIS")
print("=" * 60)

all_columns = {name: set(info.get('columns', [])) for name, info in datasets.items() 
               if info.get('exists') and 'columns' in info}

if len(all_columns) > 0:
    # Intersection of all datasets
    shared_all = set.intersection(*all_columns.values()) if all_columns else set()
    
    if shared_all:
        print(f"\n✓ Columns in ALL datasets ({len(shared_all)}):")
        for col in sorted(shared_all):
            if col in STANDARD_COLUMNS:
                unit = STANDARD_COLUMNS[col]['unit']
                desc = STANDARD_COLUMNS[col]['description']
                print(f"  • {col:20} [{unit:12}] - {desc}")
            else:
                print(f"  • {col:20} (non-standard)")
    
    # Columns in 2+ datasets
    from collections import Counter
    col_counts = Counter()
    for cols in all_columns.values():
        col_counts.update(cols)
    
    shared_2plus = {col for col, count in col_counts.items() if count >= 2}
    unique_cols = {col for col, count in col_counts.items() if count == 1}
    
    if shared_2plus - shared_all:
        print(f"\n• Columns in 2+ datasets ({len(shared_2plus - shared_all)}):")
        for col in sorted(shared_2plus - shared_all):
            datasets_with = [name for name, cols in all_columns.items() if col in cols]
            print(f"  - {col:25} in: {', '.join(datasets_with)}")
    
    if unique_cols:
        print(f"\n• Unique columns (dataset-specific, {len(unique_cols)} total):")
        for dataset_name, cols in all_columns.items():
            unique = sorted(cols & unique_cols)
            if unique:
                print(f"  {dataset_name}:")
                for col in unique[:15]:  # Show first 15 per dataset
                    print(f"    - {col}")
                if len(unique) > 15:
                    print(f"    ... and {len(unique) - 15} more")
else:
    print("\n⚠ No output files found to analyze")

# Summary table
print("\n" + "=" * 60)
print("FORMAT SUMMARY TABLE")
print("=" * 60)
print(f"{'Dataset':<18} {'Files':<8} {'Total':<8} {'Standard':<12} {'Source-Specific':<15}")
print("-" * 65)

for name, info in datasets.items():
    if info.get('exists') and 'columns' in info:
        cols = info['columns']
        file_count = info.get('file_count', 0)
        standard_count = len([c for c in cols if c in standard_cols])
        source_count = len([c for c in cols if c not in standard_cols and c != 'time'])
        print(f"{name:<18} {file_count:<8} {len(cols):<8} {standard_count:<12} {source_count:<15}")
    elif info.get('exists'):
        print(f"{name:<18} {'ERROR':<8} {'-':<8} {'-':<12} {'-':<15}")
    else:
        print(f"{name:<18} {'-':<8} {'-':<8} {'-':<12} {'-':<15}")

print("\n" + "=" * 60)

DATA FORMAT ANALYSIS

Standard columns (from config.conventions): 9
  time, temp, dewpoint, wind_speed, wind_gust, wind_dir, precip, precip_type, station

OUTPUT FILE COLUMNS

ASOS 1-Min (asos_1min_NYC.csv):
  Files: 3, Columns: 9
  ✓ Standard (8): time, temp, dewpoint, wind_speed, wind_gust, wind_dir, precip, precip_type
  • Source-specific (1): wind_gust_dir

NOAA Daily (noaa_daily_KJFK.csv):
  Files: 3, Columns: 12
  ✓ Standard (4): time, precip, temp, wind_speed
  • Source-specific (8): snow, snow_depth, temp_max, temp_min, wind_gust_2min, wind_gust_5sec, wind_dir_2min, wind_dir_5sec

NYC Mesonet (mesonet_BRON.csv):
  Files: 4, Columns: 37
  ✓ Standard (7): time, temp, dewpoint, precip, wind_speed, wind_gust, wind_dir
  • Source-specific (30): temp_9m, apparent_temperature, relative_humidity, precip_local, precip_max_intensity, precip_1hr, wind_speed_stddev_prop, wind_direction_stddev_prop, avg_wind_speed_sonic, max_wind_speed_sonic
    ... and 20 more

SHARED COLUMNS ANALYSIS

✓ C

## Summary

In [ ]:
# Print summary
print("\n" + "=" * 60)
print("PROCESSING SUMMARY")
print("=" * 60)

for name, status in results.items():
    if status is True:
        print(f"✓ {name}: SUCCESS")
    elif status is False:
        print(f"✗ {name}: FAILED")
    elif status is None:
        print(f"⚠ {name}: SKIPPED (no input data)")

print("\n" + "=" * 60)

# Check output files
if OUTPUT_DIR.exists():
    print(f"\nOutput directory: {OUTPUT_DIR}")
    
    output_dirs = {
        'ASOS 1-Min': OUTPUT_DIR / 'asos_1min',
        'NOAA Daily': OUTPUT_DIR / 'noaa_daily',
        'NYC Mesonet': OUTPUT_DIR / 'mesonet'
    }
    
    for name, out_dir in output_dirs.items():
        if out_dir.exists():
            csv_files = list(out_dir.glob("*.csv"))
            if csv_files:
                print(f"\n{name}:")
                for f in sorted(csv_files)[:10]:  # Show first 10
                    size_mb = f.stat().st_size / (1024 * 1024)
                    print(f"  ✓ {f.name} ({size_mb:.1f} MB)")
                if len(csv_files) > 10:
                    print(f"  ... and {len(csv_files) - 10} more files")
            else:
                print(f"\n{name}: No output files found")
        else:
            print(f"\n{name}: Output directory does not exist")

print("\n" + "=" * 60)


PROCESSING SUMMARY
⚠ asos_1min: SKIPPED (no input data)
⚠ noaa_daily: SKIPPED (no input data)
⚠ mesonet: SKIPPED (no input data)


Output directory: /Users/drorjac/OpenMesh_pynncml/src/data/output

ASOS 1-Min:
  ✓ asos_1min_JFK.csv (92.8 MB)
  ✓ asos_1min_LGA.csv (76.0 MB)
  ✓ asos_1min_NYC.csv (75.0 MB)

NOAA Daily:
  ✓ noaa_daily_KJFK.csv (0.1 MB)
  ✓ noaa_daily_KLGA.csv (0.1 MB)
  ✓ noaa_daily_KNYC.csv (0.1 MB)

NYC Mesonet:
  ✓ mesonet_BKLN.csv (62.9 MB)
  ✓ mesonet_BRON.csv (63.2 MB)
  ✓ mesonet_MANH.csv (58.6 MB)
  ✓ mesonet_QUEE.csv (59.3 MB)
  ✓ mesonet_metadata.csv (0.0 MB)

